In [ ]:
# CAMERA INITIALIZATION

import cv2
import numpy as py
cap = cv2.VideoCapture(1)

# if not cap.isOpened():
#     print("not open")
#     exit()


# print("camera connected")

while True:
    ret, frame = cap.read()

    if not ret:
        print("Error: COULD NOT SEE CAMERA")
        break

    cv2.imshow("Camera Test", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
         break

cap.release()
cv2.destroyAllWindows()
 


              LIGHTWEIGHT VISUAL SERVOING

This system implements classial computer vision + lightweight machine learning + robot arm control.

                      LOW-RES CAMERA
                            |
                 CLASSICAL COMPUTER VISION
                            |
                 TRADITIONAL ML CLASSIFIER
                            |
                    OBJECT LOCALIZATION
                            |
                       ROBOT CONTROL
                            


VERSION 1 -: OpenCV image processing and geometric rules with no ML model
VERRSION 2 -: The same pipeline but i will replace the classification rules with HOG+SVM or Random Forest.

to compare accuracy, execution time, memory use and robustness.


-> BGR(Blue, Green and Red) and HSV(Hue, Saturation and Value), are 2 different ways of arranging and adopting pixel data in opencv, and HSV is vastly a more recognizable option for colour segementation cause it focuses more on the isolation of colour information from brightness.

BGR is the default colour code when reading colour data in opencv,it is exactly like RGB, but channel order reversed. It deals with colour variation from a range of 0 to 255 for each colour chanel, but is more susceptible to variations in lightning for example if a shadow is cast on the colour, it can change the readings

HSV: It works well like the human senses, it breaks a pixel into 3 seperate concepts;
HUE, From 0 to 179: it represents an actual shade or colour type, it remains very stable across various lighting shifts or changes.
SATURATION,  From 0 to 255: measures the purity, where 0 is the completely faded(white/grey) and 255 is the richest and most vivid version of that color.
VALUE, From 0 to 255: it represents the brightness or luminance. A value of 0 which is pitchh black and 255 which is fully illuminated.



In [ ]:
# SEGMENTATION FOR 3 COLOURS BLUE, RED AND YELLOW.


import cv2
import numpy as np

cap = cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Example blue threshold: tune for your block and lighting.


    # Blue
    blue_mask = cv2.inRange(
        hsv, np.array([90, 70, 40]),
        np.array([135, 255, 255])
    )

    # Red: hue wraps around zero, so use two ranges
    red_mask1 = cv2.inRange(
        hsv, np.array([0, 70, 40]),
        np.array([10, 255, 255])
    )
    red_mask2 = cv2.inRange(
        hsv, np.array([170, 70, 40]),
        np.array([179, 255, 255])
    ) #from min value sof HSV to max values
    red_mask = cv2.bitwise_or(red_mask1, red_mask2)

    # Yellow
    yellow_mask = cv2.inRange(
        hsv, np.array([20, 70, 40]),
        np.array([35, 255, 255])
    )
    mask = cv2.bitwise_or(blue_mask, red_mask)
    mask = cv2.bitwise_or(mask, yellow_mask)

    

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_OPEN, kernel
    )
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_CLOSE, kernel
    )

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < 100:
            continue

        moments = cv2.moments(contour)
        if moments["m00"] == 0:
            continue

        u = int(moments["m10"] / moments["m00"])
        v = int(moments["m01"] / moments["m00"])

        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(frame, (x, y),
                      (x + w, y + h), (0, 255, 0), 2)
        cv2.circle(frame, (u, v), 4, (0, 0, 255), -1)
        cv2.putText(frame, f"Block ({u},{v})",
                    (x, max(y - 8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, (0, 255, 0), 1)

    cv2.imshow("Block detector", frame)
    cv2.imshow("Binary mask", mask)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


[ WARN:0@11626.952] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] error count: 66
[mjpeg @ 0xbe0ec40] error y=0 x=11
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] error count: 65
[mjpeg @ 0xbe0ec40] error y=0 x=10
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] error count: 68
[mjpeg @ 0xbe0ec40] error y=0 x=10
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] error count: 66
[mjpeg @ 0xbe0ec40] error y=0 x=13
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 5
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 6
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overread 8
[mjpeg @ 0xbe0ec40] overr

next to seperate the 3  shades of red for our objects below.
first we check for only red and yellow colour 

In [ ]:

import cv2
import numpy as np

cap = cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam")

while True:
    ret, frame = cap.read()
    if not ret:
        break


    # Convert the camera image from BGR to HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Red shade range 1
    red_mask1 = cv2.inRange(
        hsv,
        np.array([0, 70, 40]),
        np.array([10, 255, 255])
    )

    # Red shade range 2
    red_mask2 = cv2.inRange(
        hsv,
        np.array([170, 70, 40]),
        np.array([179, 255, 255])
    )

    # Combine the two red hue ranges
    red_mask = cv2.bitwise_or(red_mask1, red_mask2)

    # Yellow container
    yellow_mask = cv2.inRange(
        hsv,
        np.array([20, 70, 40]),
        np.array([35, 255, 255])
    )

    # Display each mask separately
    cv2.imshow("Red mask", red_mask)
    cv2.imshow("Yellow mask", yellow_mask)
    cv2.imshow("Original image", frame)

    def show_hsv(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            pixel = hsv[y, x]
            print(
                f"Pixel ({x}, {y}) ->"
                f"H = {pixel[0]}, "
                f"S = {pixel[1]}, "
                f"V = {pixel[2]}"
            )
    cv2.setMouseCallback("Original image", show_hsv)


    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
# COLOUR SEGMENTATION FOR RED AND YELLOW 


import cv2
import numpy as np

cap = cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam")

while True:
    ret, frame = cap.read()
    if not ret:
        break


    # Convert the camera image from BGR to HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Red shade range 1
    red_mask1 = cv2.inRange(
        hsv,
        np.array([0, 70, 40]),
        np.array([10, 255, 255])
    )

    # Red shade range 2
    red_mask2 = cv2.inRange(
        hsv,
        np.array([170, 70, 40]),
        np.array([179, 255, 255])
    )

    # Combine the two red hue ranges
    red_mask = cv2.bitwise_or(red_mask1, red_mask2)

    # Yellow container
    yellow_mask = cv2.inRange(
        hsv,
        np.array([20, 70, 40]),
        np.array([35, 255, 255])
    )

    # Display each mask separately
    cv2.imshow("Red mask", red_mask)
    cv2.imshow("Yellow mask", yellow_mask)
    cv2.imshow("Original image", frame)

    def show_hsv(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            pixel = hsv[y, x]
            print(
                f"Pixel ({x}, {y}) ->"
                f"H = {pixel[0]}, "
                f"S = {pixel[1]}, "
                f"V = {pixel[2]}"
            )
    cv2.setMouseCallback("Original image", show_hsv)


    contours, _= cv2.findContours(
        red_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    for contour in contours:
        area = cv2.contourArea(contour)

        if area < 100:
            continue

        x, y, w, h = cv2.boundingRect(contour)

        aspect_ratio = w / float(h)
        perimeter = cv2.arcLength(contour, True)


        circularity = (
            4 * np.pi * area / (perimeter ** 2)
        
        if perimeter > 0 else 0
        )

        hull_area = cv2.contourArea(
            cv2.convexHull(contour)
        )

        solidity = (
            area / hull_area if hull_area > 0 else 0
        )

        print(
        f"Area={area:.0f}, "
        f"Aspect ratio={aspect_ratio:.2f}, "
        f"Circularity={circularity:.2f}, "
        f"Solidity={solidity:.2f}"
    )



    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


In [25]:

import cv2
import numpy as np
import csv
import os

CAMERA_INDEX = 1
CSV_FILE = "red_objects.csv"

LABELS = {
    ord("1"): "container_red",
    ord("2"): "screwdriver",
    ord("3"): "safety_cutter",
}

FEATURE_NAMES = [
    "area",
    "width",
    "height",
    "aspect_ratio",
    "perimeter",
    "circularity",
    "solidity",
    "extent",
    "label",
]

# Create the CSV header if the file does not exist
if not os.path.exists(CSV_FILE):
    with open(CSV_FILE, "w", newline="") as file:
        writer = csv.DictWriter(
            file, fieldnames=FEATURE_NAMES
        )
        writer.writeheader()


def extract_features(contour):
    area = cv2.contourArea(contour)

    x, y, width, height = cv2.boundingRect(contour)

    perimeter = cv2.arcLength(contour, True)

    circularity = (
        4 * np.pi * area / (perimeter ** 2)
        if perimeter > 0 else 0
    )

    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)

    solidity = (
        area / hull_area if hull_area > 0 else 0
    )

    box_area = width * height

    extent = (
        area / box_area if box_area > 0 else 0
    )

    return {
        "area": area,
        "width": width,
        "height": height,
        "aspect_ratio": width / height,
        "perimeter": perimeter,
        "circularity": circularity,
        "solidity": solidity,
        "extent": extent,
    }


cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    raise RuntimeError("Could not open camera")

print("Place ONE red object in view.")
print("1 = red container")
print("2 = screwdriver")
print("3 = safety cutter")
print("Q = quit")

while True:
    ret, frame = cap.read()

    if not ret:
        print("Could not read camera frame")
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Red wraps around the HSV hue boundary
    mask1 = cv2.inRange(
        hsv, np.array([0, 70, 40]),
        np.array([10, 255, 255])
    )
    mask2 = cv2.inRange(
        hsv, np.array([170, 70, 40]),
        np.array([179, 255, 255])
    )

    red_mask = cv2.bitwise_or(mask1, mask2)

    kernel = np.ones((3, 3), np.uint8)
    red_mask = cv2.morphologyEx(
        red_mask, cv2.MORPH_OPEN, kernel
    )
    red_mask = cv2.morphologyEx(
        red_mask, cv2.MORPH_CLOSE, kernel
    )

    contours, _ = cv2.findContours(
        red_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Find the largest red contour
    candidates = [
        c for c in contours
        if cv2.contourArea(c) >= 100
    ]

    largest = (
        max(candidates, key=cv2.contourArea)
        if candidates else None
    )

    display = frame.copy()

    if largest is not None:
        x, y, w, h = cv2.boundingRect(largest)

        cv2.rectangle(
            display, (x, y), (x + w, y + h),
            (0, 255, 0), 2
        )

        cv2.putText(
            display, "Detected object",
            (x, max(y - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5,
            (0, 255, 0), 2
        )

    cv2.imshow("Object data collection", display)
    cv2.imshow("Red mask", red_mask)

    key = cv2.waitKey(1) & 0xFF

    if key in LABELS and largest is not None:
        features = extract_features(largest)
        features["label"] = LABELS[key]

        with open(CSV_FILE, "a", newline="") as file:
            writer = csv.DictWriter(
                file, fieldnames=FEATURE_NAMES
            )
            writer.writerow(features)

        print("Saved:", features)

    elif key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


[ WARN:0@12850.335] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index


Place ONE red object in view.
1 = red container
2 = screwdriver
3 = safety cutter
Q = quit


[mjpeg @ 0xbbe4180] overread 8
[mjpeg @ 0xbbe4180] overread 8
[mjpeg @ 0xbbe4180] overread 8
[mjpeg @ 0xbbe4180] error count: 73
[mjpeg @ 0xbbe4180] error y=0 x=12
[mjpeg @ 0xbbe4180] overread 8
[mjpeg @ 0xbbe4180] overread 8
[mjpeg @ 0xbbe4180] error count: 65
[mjpeg @ 0xbbe4180] error y=0 x=12
[mjpeg @ 0xbbe4180] overread 8
[mjpeg @ 0xbbe4180] overread 8
[mjpeg @ 0xbbe4180] overread 5
[mjpeg @ 0xbbe4180] error count: 64
[mjpeg @ 0xbbe4180] error y=0 x=13
[mjpeg @ 0xbbe4180] overread 8
[mjpeg @ 0xbbe4180] overread 8
[mjpeg @ 0xbbe4180] overread 8


KeyboardInterrupt: 

In [ ]:

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

# 1. Load and inspect the dataset
data = pd.read_csv("/home/goodness/Documents/PYTHON_LINK/red_objects.csv")

print("Dataset shape:", data.shape)
print("\nSamples per class:")
print(data["label"].value_counts())

features = [
    "area",
    "width",
    "height",
    "aspect_ratio",
    "perimeter",
    "circularity",
    "solidity",
    "extent",
]

# 2. Prepare inputs (X) and target labels (y)
X = data[features].apply(pd.to_numeric, errors="coerce")
y = data["label"]

# Remove rows with missing feature values or labels
valid = X.notna().all(axis=1) & y.notna()
X = X.loc[valid]
y = y.loc[valid]

if y.nunique() < 2:
    raise ValueError("The dataset needs at least two object classes.")

if y.value_counts().min() < 2:
    raise ValueError(
        "Each class needs at least two samples for this split."
    )

# 3. Split: 75% training, 25% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

# 4. Create and train the classifier
model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42,
)

model.fit(X_train, y_train)

# 5. Evaluate on unseen test samples
predictions = model.predict(X_test)

print("\nTest accuracy:",
      round(accuracy_score(y_test, predictions), 3))

print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(
    y_test,
    predictions,
    labels=model.classes_,
))
print("Class order:", list(model.classes_))

# 6. Save model and feature order
joblib.dump(
    {"model": model, "features": features},
    "red_object_classifier.joblib",
)

print("\nSaved: red_object_classifier.joblib")


In [ ]:
import pandas as pd

columns = [
    "area",
    "width",
    "height",
    "aspect_ratio",
    "perimeter",
    "circularity",
    "solidity",
    "extent",
    "label",
]

data = pd.read_csv(
    "red_objects.csv",
    header=None,
    names=columns
)

print(data.head())
print("\nSamples per class:")
print(data["label"].value_counts())

In [ ]:
import pandas as pd

data = pd.read_csv("red_objects.csv")

print(data.columns.tolist())
print(data.head())